# 07. 시간순 거래 재생과 배치 추론

Spark가 계산한 Test 거래의 과거 이력 특징을 API와 같은 추론 서비스에 전달한다. 이력을 온라인으로 갱신하는 기능은 후속 작업으로 구분한다. 10,000건씩 step 오름차순으로 재생하고 정답을 제외한 예측 결과를 저장한다. 동일 step 내부 순서는 실제 발생 순서로 해석하지 않는다.

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from fds_agent.models.inference import FraudInference, TransactionFeatures  # noqa: E402

model_run = "20261119T073653Z-1d0e70df"
service = FraudInference(ROOT / "artifacts/models", model_run)
source = json.loads((ROOT / service.selection["source_manifest"]).read_text())
assert source["verified"]
expected = next(s for s in source["splits"] if s["split"] == "test")
fields = list(TransactionFeatures.model_fields)
data = pd.read_parquet(
    ROOT / source["parquet_path"] / "split=test",
    columns=fields + ["nameOrig", "nameDest", "isFraud"],
)
assert len(data) == expected["rows"]
data = data.sort_values("step", kind="stable").reset_index(drop=True)
assert data.step.is_monotonic_increasing
# 동일 step 내부 순서는 실제 발생 순서로 해석하지 않는다.
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
output = ROOT / "artifacts/reports" / run_id
output.mkdir(parents=True)
print("Replay rows:", len(data), "Run:", run_id)

Replay rows: 1293523 Run: 20261119T094739Z-1dbbe217


## 배치 결과 저장과 고정 평가 결과 대조

단건 API와 배치 추론의 점수를 비교하고 전체 결과 건수와 혼동행렬이 이전 고정 Test 평가와 일치하는지 확인한다. 결과에 맞춰 모델이나 임계값을 변경하지 않는다.

In [2]:
batch_size = 10000
summaries = []
reference = json.loads(
    (ROOT / "artifacts/reports/20261119T075635Z-6a886eab/manifest.json").read_text()
)
counts = dict(tn=0, fp=0, fn=0, tp=0)
for batch_id, start in enumerate(range(0, len(data), batch_size)):
    batch = data.iloc[start : start + batch_size]
    requests = [TransactionFeatures.model_validate(row) for row in batch[fields].to_dict("records")]
    predictions = service.predict_batch(requests)
    if batch_id == 0:
        # 단건 API와 배치 경로의 점수 일치를 확인한다.
        for item, result in zip(requests[:3], predictions[:3]):
            assert np.isclose(service.predict(item).risk_score, result.risk_score, atol=1e-12)
    result = batch[["step", "nameOrig", "nameDest"]].copy()
    result["replay_sequence"] = np.arange(start, start + len(batch))
    result["risk_score"] = [p.risk_score for p in predictions]
    result["is_fraud"] = [p.is_fraud for p in predictions]
    result["threshold"] = service.threshold
    result["model_run_id"] = service.run_id
    result.to_parquet(output / f"predictions-{batch_id:04d}.parquet", index=False)
    # 정답은 추론 요청에 포함하지 않고 결과 대조에만 사용한다.
    truth = batch.isFraud.to_numpy() == 1
    guessed = result.is_fraud.to_numpy()
    counts["tn"] += int((~truth & ~guessed).sum())
    counts["fp"] += int((~truth & guessed).sum())
    counts["fn"] += int((truth & ~guessed).sum())
    counts["tp"] += int((truth & guessed).sum())
    summaries.append(
        dict(
            batch=batch_id,
            rows=len(batch),
            min_step=int(batch.step.min()),
            max_step=int(batch.step.max()),
            alerts=int(guessed.sum()),
        )
    )
assert sum(counts.values()) == expected["rows"]
assert counts == {k: reference["metrics"][k] for k in counts}
manifest = dict(
    run_id=run_id,
    model_run_id=model_run,
    source=service.selection["source_manifest"],
    rows=len(data),
    batches=len(summaries),
    threshold=service.threshold,
    counts=counts,
    verified=True,
    mode="offline replay of Spark precomputed prior-step features",
    historical_features_updated_online=False,
)
(output / "manifest.json").write_text(json.dumps(manifest, indent=2))
summary = pd.DataFrame(summaries)
summary.to_csv(output / "batches.csv", index=False)
display(summary.head())
print(manifest)

,batch,rows,min_step,max_step,alerts
0,0,10000,355,355,1
1,1,10000,355,355,4
2,2,10000,355,355,1
3,3,10000,355,355,0
4,4,10000,355,356,3


{'run_id': '20261119T094739Z-1dbbe217', 'model_run_id': '20261119T073653Z-1d0e70df', 'source': 'artifacts/features/paysim_temporal/20261117T090124Z-c769f595/manifest.json', 'rows': 1293523, 'batches': 130, 'threshold': 0.006350361245177384, 'counts': {'tn': 1289265, 'fp': 0, 'fn': 1, 'tp': 4257}, 'verified': True, 'mode': 'offline replay of Spark precomputed prior-step features', 'historical_features_updated_online': False}
